In [1]:
import numpy as np
from manifold3d import Manifold, Mesh
import k3d
import trimesh
from pathlib import Path
import polars as pl

In [2]:
def create_box_manifold(width, length, height, z_offset=0.0):
    """Generates a Manifold object for a solid rectangular box."""
    # 1. Define 8 vertices for a straight box
    vertices = np.array([
        # Bottom 4 vertices (z = z_offset)
        [-width / 2, -length / 2, z_offset],  # 0: Bottom-left
        [ width / 2, -length / 2, z_offset],  # 1: Bottom-right
        [ width / 2,  length / 2, z_offset],  # 2: Top-right
        [-width / 2,  length / 2, z_offset],  # 3: Top-left
        
        # Top 4 vertices (z = height + z_offset)
        [-width / 2, -length / 2, height + z_offset],  # 4: Top-left (Upper)
        [ width / 2, -length / 2, height + z_offset],  # 5: Top-right (Upper)
        [ width / 2,  length / 2, height + z_offset],  # 6: Bottom-right (Upper)
        [-width / 2,  length / 2, height + z_offset],  # 7: Bottom-left (Upper)
    ], dtype=np.float32)

    # 2. Define 12 triangular faces
    faces = np.array([
        [0, 2, 1], [0, 3, 2],  # Bottom
        [4, 5, 6], [4, 6, 7],  # Top
        [0, 1, 5], [0, 5, 4],  # Front
        [1, 2, 6], [1, 6, 5],  # Right
        [2, 3, 7], [2, 7, 6],  # Back
        [3, 0, 4], [3, 4, 7],  # Left
    ], dtype=np.int32)

    # 3. Construct the native Manifold object
    mesh = Mesh(vert_properties=vertices, tri_verts=faces)
    return Manifold(mesh)

def create_frustum_manifold(bottom_w, bottom_l, top_w, top_l, height, z_offset=0.0):
    """
    Generates a Manifold object for a 4-sided frustum.
    bottom_w, bottom_l: bottom width and length
    top_w, top_l: top length and width
    """
    # 1. Define 8 vertices for the frustum
    vertices = np.array([
        [-bottom_w / 2, -bottom_l / 2, z_offset],  # 0: Bottom-left
        [ bottom_w / 2, -bottom_l / 2, z_offset],  # 1: Bottom-right
        [ bottom_w / 2,  bottom_l / 2, z_offset],  # 2: Top-right
        [-bottom_w / 2,  bottom_l / 2, z_offset],  # 3: Top-left
        
        [-top_w / 2, -top_l / 2, height + z_offset],  # 4: Top-left (Upper)
        [ top_w / 2, -top_l / 2, height + z_offset],  # 5: Top-right (Upper)
        [ top_w / 2,  top_l / 2, height + z_offset],  # 6: Bottom-right (Upper)
        [-top_w / 2,  top_l / 2, height + z_offset],  # 7: Bottom-left (Upper)
    ], dtype=np.float32)

    # 2. Define 12 triangular faces for the frustum
    faces = np.array([
        [0, 2, 1], [0, 3, 2],  # bottom face (facing -z)
        [4, 5, 6], [4, 6, 7],  # top face (facing +z)
        [0, 1, 5], [0, 5, 4],  # front face (facing -y)
        [1, 2, 6], [1, 6, 5],  # right face (facing +x)
        [2, 3, 7], [2, 7, 6],  # back face (facing +y)
        [3, 0, 4], [3, 4, 7],  # left face (facing -x)
    ], dtype=np.int32)

    # 3. Construct the native Manifold object
    mesh = Mesh(vert_properties=vertices, tri_verts=faces)
    return Manifold(mesh)

def create_hollow_box(in_w, in_l, h, thickness, z_offset=0.0):
    """
    Creates a hollow rectangular box (tube) defined by its INNER dimensions.
    Top and bottom are fully open.

    in_w, in_l: inner width and length
    h: total height of the box
    thickness: wall thickness
    """
    # 1. Calculate outer dimensions directly (just add thickness to both sides)
    out_w = in_w + (2 * thickness)
    out_l = in_l + (2 * thickness)
    out_h = h

    # 2. Calculate inner cutting dimensions
    # Extend the inner box by 1 unit on top and bottom to ensure a clean Boolean cut
    z_extend = 1.0
    ext_in_h = h + (2 * z_extend)

    # 3. Build Outer and Inner Manifolds
    outer = create_box_manifold(out_w, out_l, out_h, z_offset=z_offset)

    # Push the inner box down by z_extend so it cuts completely through the floor and ceiling
    inner = create_box_manifold(in_w, in_l, ext_in_h, z_offset=z_offset - z_extend)

    # 4. Native Boolean Difference
    hollow_box = outer - inner

    return hollow_box


def create_collimator_manifold(
    l_top, h_nozzle, w_pinhole, w_wall, l_bottom_inner, l_bottom_outer, h_body, h_box
):
    """
    Creates a frustum collimator with specified dimensions.

    l_top: length of the top opening (pinhole)
    h_nozzle: height of the nozzle section
    w_pinhole: width of the pinhole
    w_wall: wall thickness
    l_bottom_inner: inner length of the bottom opening
    l_bottom_outer: outer length of the bottom opening
    h_body: height of the body section
    h_box: height of the box section
    """

    # Create the main frustum body
    frustum_manifold_a = create_frustum_manifold(
        l_bottom_outer, l_bottom_outer, l_top, l_top, h_nozzle + h_body, z_offset=h_box
    )
    frustum_manifold_b = create_frustum_manifold(
        w_pinhole, w_pinhole, l_top, l_top, h_nozzle, z_offset=h_body + h_box
    )
    frustum_manifold_c = create_frustum_manifold(
        l_bottom_inner, l_bottom_inner, w_pinhole, w_pinhole, h_body, z_offset=h_box
    )
    base_box_manifold = create_hollow_box(
        l_bottom_outer - 2 * w_wall, l_bottom_outer - 2 * w_wall, h_box, w_wall
    )
    # basde_box_manifold = base_box_manifold.translate([0, 0, h_nozzle + h_body])
    # collimator_manifold = frustum_manifold_a - frustum_manifold_b - frustum_manifold_c + base_box_manifold
    collimator_manifold = (
        frustum_manifold_a - frustum_manifold_b - frustum_manifold_c + base_box_manifold
    )
    return collimator_manifold


collimator_manifold = create_collimator_manifold(
    l_top=5.04,
    h_nozzle=10.74,
    w_pinhole=0.556,
    w_wall=2.03,
    l_bottom_inner=50.0,
    l_bottom_outer=56.064,
    h_body=25.0,
    h_box=23.5,
)

mesh_data = collimator_manifold.to_mesh()
vertices = mesh_data.vert_properties[:, :3].astype(np.float32)
indices = mesh_data.tri_verts.astype(np.uint32)

# --- NEW: Feature Edge Extraction ---
# 2. Load the raw vertices and faces into Trimesh (process=False prevents altering the vertex order)
tm = trimesh.Trimesh(vertices=vertices, faces=indices, process=False)

# Define angle threshold (e.g., 30 degrees) to capture sharp corners. 
# Because collimator geometry usually consists of distinct planar faces, 30-45 degrees is very reliable.
threshold = np.radians(30)

# Identify edges where the angle between adjacent faces exceeds the threshold
feature_mask = tm.face_adjacency_angles > threshold
feature_edges = tm.face_adjacency_edges[feature_mask]
# ------------------------------------

# 3. Initialize the K3D plot
plot = k3d.plot(name="Collimator Mesh")

# 4. Create the solid K3D mesh object
# Set wireframe=False to hide the internal triangulation
mesh_vis = k3d.mesh(
    vertices,
    indices,
    color=0x00A0FF,  
    wireframe=False,  
    side="double",
    opacity=1  # Slight transparency helps visualize internal septa
)

# 5. Create the K3D lines object for the feature edges
# K3D's 'segment' type expects a flat 1D array of [start, end, start, end...] indices
line_vis = k3d.lines(
    vertices,
    indices=feature_edges.flatten().astype(np.uint32),
    indices_type='segment',
    color=0x000000,   # Black contour lines
    line_width=2.0    # Adjust thickness as needed
)

# 6. Add both objects to the plot and display
plot += mesh_vis
plot += line_vis
plot.display()

/Users/fanghan/.local/share/mise/installs/python/3.14.4/lib/python3.14/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "uint32" does not match required type "float32". A coerced copy has been created.
  warnings.warn(


Output()

In [3]:
h_body = 25.0
h_box = 23.5
# load the csv data using polars
csv_dir = Path('/Users/fanghan/Work/RPIL/rpil-research-note/brain-spect-geometry/python')
csv_filename = 'BrainSPECT_Point_Cloud.007.25mmx0.556mm_pinhole.csv'
csv_pl_df = pl.read_csv(csv_dir / csv_filename)
print(f"Shape of CSV DataFrame: {csv_pl_df.shape}")
print(f"Columns: {csv_pl_df.columns}")

collimator_manifold = create_collimator_manifold(
    l_top=10.74,
    h_nozzle=5.04,
    w_pinhole=0.556,
    w_wall=2.03,
    l_bottom_inner=50.0,
    l_bottom_outer=56.064,
    h_body=h_body,
    h_box=h_box,
)

# collimator_mesh_data = collimator_manifold.to_mesh()
# collimator_vertices = (
#    collimator_mesh_data.vert_properties[:, :3].astype(np.float32)
# )
# collimator_indices = collimator_mesh_data.tri_verts.astype(np.uint32)

collimator_manifold = collimator_manifold.translate([0.0, 0.0, -(h_body+h_box)])  # Shift down by sum of body and box heights in Z
collimator_manifold = collimator_manifold.rotate([0.0, -90.0, 0.0])  # Rotate around X-axis by -90 degrees

# permutate the x, y, z order of the pinhole points to match default k3d coordinate system
# The raw order assume y is up, but k3d assumes z is up. So we need to swap y and z.
pinhole_points = (
    csv_pl_df.select(["Pinhole_x", "Pinhole_y", "Pinhole_z"])
    .to_numpy()
    .astype(np.float32)
)
pinhole_points = pinhole_points[:, [0, 2, 1]]

plot = k3d.plot(name="Single Collimator")

# pinhole_k3d = k3d.points(
#     pinhole_points, point_size=4, color=0x00FF00, shader="flat", name="Pinhole Points"
# )
# plot += pinhole_k3d

# calculate the azimuth and elevation angles of the vector from the origin to the pinhole point
elevation = np.arctan2(pinhole_points[:, 2], np.sqrt(pinhole_points[:, 0]**2 + pinhole_points[:, 1]**2))
elevation = (-elevation + 2 * np.pi) % (2 * np.pi)  # convert elevation angle to 0 to 2pi range
# convert azimuthal angle to 0 to 2pi range
azimuth = (np.arctan2(pinhole_points[:, 1], pinhole_points[:, 0]) + 2 * np.pi) % (2 * np.pi)


for az, el, point in list(zip(azimuth, elevation, pinhole_points)):  # 
    transformed_collimator_manifold = collimator_manifold.rotate([0.0, np.degrees(el), np.degrees(az)])  # Rotate around Z-axis by azimuth
    transformed_collimator_manifold = transformed_collimator_manifold.translate(point)
    # transformed_collimator_manifold = transformed_collimator_manifold.rotate([0.0, 0.0, np.degrees(az)])  # Rotate around X-axis by -90 degrees
    transformed_collimator_mesh = transformed_collimator_manifold.to_mesh()
    transformed_vertices = transformed_collimator_mesh.vert_properties[:, :3].astype(np.float32)
    transformed_indices = transformed_collimator_mesh.tri_verts.astype(np.uint32)
    # Add the transformed collimator to the plot
    collimator_faces_k3d = k3d.mesh(
        transformed_vertices,
        transformed_indices,
        color=0x00A0FF,
        wireframe=False,
        opacity=1,
        side="double",
        depth_test=False,
        name=f"Collimator at Azimuth: {az:.2f}, Elevation: {el:.2f}",
        shader="flat",
    )
    # --- NEW: Feature Edge Extraction ---
    # 2. Load the raw vertices and faces into Trimesh (process=False prevents altering the vertex order)
    tm = trimesh.Trimesh(vertices=transformed_vertices, faces=transformed_indices, process=False)

    # Define angle threshold (e.g., 30 degrees) to capture sharp corners. 
    # Because collimator geometry usually consists of distinct planar faces, 30-45 degrees is very reliable.
    threshold = np.radians(30)

    # Identify edges where the angle between adjacent faces exceeds the threshold
    feature_mask = tm.face_adjacency_angles > threshold
    feature_edges = tm.face_adjacency_edges[feature_mask]

    collimator_silhouette_k3d = k3d.lines(
    transformed_vertices,
    indices=feature_edges.flatten().astype(np.uint32),
    indices_type='segment',
    color=0x000000,   # Black contour lines
    line_width=2.0    # Adjust thickness as needed
)

# 6. Add both objects to the plot and display
    plot += collimator_silhouette_k3d
    plot += collimator_faces_k3d

plot.camera = [
    10, 10, 10,  # 1. Camera Position (Where your eyes are in 3D space)
    0, 0, 0,     # 2. Target/Look-At Point (What you are looking at)
    0, 0, 1      # 3. Up Vector (Forces Z to be the vertical "up" direction)
]


stl_dir = Path("/Users/fanghan/Work/RPIL/rpil-research-note/brain-spect-geometry/python/stl")
stl_filename = "BrainFrame.008.Lead_Shield.STL"
stl_mesh = trimesh.load(stl_dir / stl_filename, force="mesh")

stl_verts = np.ascontiguousarray(stl_mesh.vertices.astype(np.float32))
stl_faces = np.ascontiguousarray(stl_mesh.faces.astype(np.uint32))

stl_verts = stl_verts[:, [0, 2, 1]]

# Import STL Mesh (Solid Orange)
stl_k3d = k3d.mesh(
    stl_verts,
    stl_faces,
    color=0xFF5733,
    opacity=1,
    side="double",
    wireframe=False,
    flat_shading=True,
    name="Imported STL",
)
plot += stl_k3d

plot.display()
with open("brain_spect_geometry.html", "w", encoding="utf-8") as f:
    f.write(plot.get_snapshot())


Shape of CSV DataFrame: (73, 6)
Columns: ['Pinhole_x', 'Pinhole_y', 'Pinhole_z', 'Crystal_x', 'Crystal_y', 'Crystal_z']


/Users/fanghan/.local/share/mise/installs/python/3.14.4/lib/python3.14/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "uint32" does not match required type "float32". A coerced copy has been created.
  warnings.warn(


Output()